In [57]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import numpy as np

from pyproj import Transformer
from sklearn.neighbors import BallTree
from pathlib import Path
import requests
import zipfile
from pathlib import Path

In [58]:
# Carpetas de trabajo
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
GRAPH_DIR = DATA_DIR / "graphs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

In [59]:
url_csv = "https://datos.madrid.es/dataset/202468-0-intensidad-trafico/resource/202468-294-intensidad-trafico/download/202468-294-intensidad-trafico.csv"

df_medidores = pd.read_csv(
    url_csv,
    sep=";",
    encoding="latin1"
)

df_medidores.head()

,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud
0,other,1.0,6835,18RA28PM01,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,other,9.0,1012,18RA66PM01,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,URB,10.0,5035,95013,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,URB,5.0,5579,61068,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,URB,5.0,5580,61069,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [60]:
df_medidores.columns

Index(['tipo_elem', 'distrito', 'id', 'cod_cent', 'nombre', 'utm_x', 'utm_y',
       'longitud', 'latitud'],
      dtype='object')

In [61]:
medidores = df_medidores[["id", "nombre", "utm_x", "utm_y", "longitud", "latitud"]].copy()

medidores = medidores.dropna(subset=["longitud", "latitud"])
medidores = medidores.drop_duplicates(subset=["id"])

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [62]:
print("Número de medidores:", len(medidores))
medidores.head()

Número de medidores: 5072


,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [63]:
medidores.dtypes

id            int64
nombre       object
utm_x       float64
utm_y       float64
longitud    float64
latitud     float64
dtype: object

In [64]:
# Comprobaciones de calidad de datos
print("IDs duplicados:", medidores["id"].duplicated().sum())
print("Latitud nula:", medidores["latitud"].isna().sum())
print("Longitud nula:", medidores["longitud"].isna().sum())

fuera_madrid = medidores[
    ~(
        (medidores["latitud"].between(40.30, 40.55)) &
        (medidores["longitud"].between(-3.90, -3.50))
    )
]

print("Medidores fuera de rango Madrid:", len(fuera_madrid))
fuera_madrid.head()

IDs duplicados: 0
Latitud nula: 0
Longitud nula: 0
Medidores fuera de rango Madrid: 0


,id,nombre,utm_x,utm_y,longitud,latitud


In [65]:
medidores[["latitud", "longitud"]].describe()

,latitud,longitud
count,5072.000000,5072.000000
mean,40.430447,-3.684001
std,0.039163,0.042728
min,40.332454,-3.836886
25%,40.398978,-3.712553
50%,40.431302,-3.686923
75%,40.460080,-3.656194
max,40.515611,-3.551623


## Descarga de la red viaria de Madrid con OSMnx

In [66]:
G_osm = ox.graph_from_place(
    "Madrid, Spain",
    network_type="drive",
    simplify=True
)

print("Nodos OSM:", len(G_osm.nodes))
print("Aristas OSM:", len(G_osm.edges))

Nodos OSM: 31453
Aristas OSM: 61857


## Snap de medidores a la red OSM

Cada punto medidor se asocia al nodo OSM más cercano.  
 `snap to graph + camino más corto sobre OSMnx`.

In [67]:
osm_nodes, distancias = ox.distance.nearest_nodes(
    G_osm,
    X=medidores["longitud"].values,
    Y=medidores["latitud"].values,
    return_dist=True
)

medidores["osm_node"] = osm_nodes
medidores["distancia_osm_node_m"] = distancias

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315,32636471,79.621034
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861,315259372,54.829432
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040,305399713,15.640852
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932,1672792326,40.116809
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073,119794656,14.979371


In [68]:
medidores["distancia_osm_node_m"].describe()

count    5072.000000
mean       36.635675
std        28.015551
min         0.101869
25%        16.227750
50%        28.117214
75%        50.402014
max       345.242185
Name: distancia_osm_node_m, dtype: float64

In [69]:
medidores.sort_values("distancia_osm_node_m", ascending=False).head(20)

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
1938,5268,(TACTICO)SALIDA POLIGONO N-S,434514.277528,4.468632e+06,-3.771300,40.365688,306400716,345.242185
107,4928,(TACTICO) AV. POBLADOS O-E (GIRO A ERICA),435668.604184,4.470451e+06,-3.757889,40.382164,282940163,228.695575
1394,4960,(TACTICO) ERICA N-S (CENTRO C.I.E.),435691.441140,4.470458e+06,-3.757621,40.382233,282940163,219.565346
1760,11199,Fuerzas Armadas - Ciudad Deportiva O-E - Fuerz...,448191.892131,4.481464e+06,-3.611259,40.482247,1012899036,178.720804
2325,11200,Fuerzas Armadas - Ciudad Deportiva O-E (Vía Se...,448192.910496,4.481435e+06,-3.611244,40.481993,1012899118,178.333987
4888,6876,12XC06PM01,441861.911399,4.471142e+06,-3.684994,40.388851,317771984,169.480943
1745,11191,"Av Fuerzas Armadas, 322 O-E - Av Fuerzas Armad...",447371.205461,4.481464e+06,-3.620941,40.482198,969169634,168.795375
1759,11192,"Av Fuerzas Armadas, 322 O-E (Via Servicio) - A...",447372.223825,4.481436e+06,-3.620927,40.481944,969169634,168.326941
446,9916,SINESIO DELGADO O-E (HOSPITAL CARLOS III-ENTRA...,440954.801598,4.480675e+06,-3.696566,40.474658,26205041,163.715082
445,9915,SINESIO DELGADO E-O (SALIDA TUNEL-HOSPITAL CAR...,440949.748176,4.480680e+06,-3.696627,40.474708,26205041,156.229910


## Análisis de distancias entre medidores y generación de pares candidatos

In [70]:
# Coordenadas en radianes para distancia haversine
coords = np.radians(medidores[["latitud", "longitud"]].values)

tree = BallTree(coords, metric="haversine")

# Calculamos hasta los 20 vecinos más cercanos para estudiar la distribución
K_ANALISIS = 20
distancias, indices = tree.query(coords, k=K_ANALISIS + 1)

R = 6371000  # radio tierra metros
distancias_m = distancias * R

In [71]:
resumen_vecinos = pd.DataFrame({
    "vecino_1_m": distancias_m[:, 1],
    "vecino_3_m": distancias_m[:, 3],
    "vecino_5_m": distancias_m[:, 5],
    "vecino_10_m": distancias_m[:, 10],
    "vecino_20_m": distancias_m[:, 20],
})

resumen_vecinos.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95])

,vecino_1_m,vecino_3_m,vecino_5_m,vecino_10_m,vecino_20_m
count,5072.000000,5072.000000,5072.000000,5072.000000,5072.000000
mean,60.299512,134.802462,192.218004,303.847693,479.846961
std,50.277204,83.244571,106.243956,150.109013,302.088483
min,0.000000,10.896282,15.656927,80.867585,195.699433
25%,16.317860,89.649261,130.094414,216.112762,344.834643
50%,51.730719,125.062705,172.480334,274.661836,421.074841
75%,90.709754,164.091985,225.365385,348.903363,527.182705
90%,123.780820,211.121938,299.572190,449.004668,660.410756
95%,148.631821,252.537244,367.262305,544.203109,817.526027
max,559.979456,1626.513574,1656.200417,1741.671511,4377.116647


In [72]:
radio_candidatos_m = np.percentile(distancias_m[:, 5], 90)

print("Radio de candidatos calculado:", radio_candidatos_m, "metros")

Radio de candidatos calculado: 299.5721899933185 metros


In [73]:
radio_candidatos_rad = radio_candidatos_m / R

indices_radio, distancias_radio = tree.query_radius(
    coords,
    r=radio_candidatos_rad,
    return_distance=True,
    sort_results=True
)

pares_candidatos = []

for i in range(len(medidores)):
    medidor_origen = medidores.iloc[i]
    
    for j, distancia_rad in zip(indices_radio[i], distancias_radio[i]):
        if i == j:
            continue
        
        medidor_destino = medidores.iloc[j]
        
        pares_candidatos.append({
            "id_origen": medidor_origen["id"],
            "id_destino": medidor_destino["id"],
            "distancia_directa_m": distancia_rad * R,
            "osm_node_origen": medidor_origen["osm_node"],
            "osm_node_destino": medidor_destino["osm_node"]
        })

df_pares = pd.DataFrame(pares_candidatos)

print("Número de pares candidatos:", len(df_pares))
df_pares.head()

Número de pares candidatos: 60494


,id_origen,id_destino,distancia_directa_m,osm_node_origen,osm_node_destino
0,6835,6833,20.104431,32636471,32636471
1,6835,6836,91.872471,32636471,315261895
2,6835,6837,93.934596,32636471,315264896
3,6835,6827,110.969150,32636471,315261895
4,6835,1042,116.845927,32636471,315265031


In [74]:
candidatos_por_medidor = (
    df_pares
    .groupby("id_origen")
    .size()
    .reset_index(name="num_candidatos")
)

candidatos_por_medidor["num_candidatos"].describe()


count    5060.000000
mean       11.955336
std         6.228532
min         1.000000
25%         7.000000
50%        11.000000
75%        16.000000
max        38.000000
Name: num_candidatos, dtype: float64

## Cálculo de caminos reales sobre la red OSM

In [75]:
aristas_reales = []

for _, row in tqdm(df_pares.iterrows(), total=len(df_pares)):
    id_origen = int(row["id_origen"])
    id_destino = int(row["id_destino"])

    osm_origen = int(row["osm_node_origen"])
    osm_destino = int(row["osm_node_destino"])

    distancia_directa_m = float(row["distancia_directa_m"])

    # Caso especial: dos medidores asociados al mismo nodo OSM
    if osm_origen == osm_destino:
        aristas_reales.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": 0.0,
            "tipo_conexion": "mismo_nodo_osm"
        })
        continue

    try:
        distancia_red_m = nx.shortest_path_length(
            G_osm,
            source=osm_origen,
            target=osm_destino,
            weight="length"
        )

        aristas_reales.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": float(distancia_red_m),
            "tipo_conexion": "camino_osm"
        })

    except (nx.NetworkXNoPath, nx.NodeNotFound):
        # Si no hay camino real en OSM, no se crea arista
        continue

df_aristas_reales = pd.DataFrame(aristas_reales).drop_duplicates()

print("Pares candidatos:", len(df_pares))
print("Aristas reales encontradas:", len(df_aristas_reales))

df_aristas_reales.head()

  0%|          | 0/60494 [00:00<?, ?it/s]

100%|██████████| 60494/60494 [01:26<00:00, 698.32it/s] 


Pares candidatos: 60494
Aristas reales encontradas: 60251


,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion
0,6835,6833,32636471,32636471,20.104431,0.000000,mismo_nodo_osm
1,6835,6836,32636471,315261895,91.872471,6655.017406,camino_osm
2,6835,6837,32636471,315264896,93.934596,2248.920678,camino_osm
3,6835,6827,32636471,315261895,110.969150,6655.017406,camino_osm
4,6835,1042,32636471,315265031,116.845927,2583.076553,camino_osm


## Validación de aristas
Se calcula el factor de rodeo:

\[
factor\_rodeo = \frac{distancia\_red\_m}{distancia\_directa\_m}
\]

Este factor permite detectar conexiones donde dos medidores están cerca en línea recta, pero el camino real por carretera es mucho más largo.  
Estas aristas no se eliminan automáticamente: se marcan para revisión.

In [76]:
df_aristas_reales["factor_rodeo"] = np.where(
    df_aristas_reales["distancia_directa_m"] > 0,
    df_aristas_reales["distancia_red_m"] / df_aristas_reales["distancia_directa_m"],
    np.nan
)

# Si están en el mismo nodo OSM, consideramos factor de rodeo 0
df_aristas_reales.loc[
    df_aristas_reales["tipo_conexion"] == "mismo_nodo_osm",
    "factor_rodeo"
] = 0

df_aristas_reales[[
    "distancia_directa_m",
    "distancia_red_m",
    "factor_rodeo"
]].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])

,distancia_directa_m,distancia_red_m,factor_rodeo
count,60251.000000,60251.000000,60251.000000
mean,182.030328,634.300037,4.189254
std,77.679277,1101.004101,12.548082
min,0.000000,0.000000,0.000000
50%,190.990925,332.662946,1.647012
75%,248.275283,588.988119,3.171711
90%,279.856331,1315.042019,7.481318
95%,289.919282,2565.392329,14.442797
99%,297.865966,5247.698913,47.562994
max,299.571214,13963.539403,734.950412


In [77]:
# Umbrales 
umbral_distancia_red = df_aristas_reales["distancia_red_m"].quantile(0.95)
umbral_rodeo = df_aristas_reales["factor_rodeo"].quantile(0.95)

print("Umbral distancia red P95:", umbral_distancia_red)
print("Umbral rodeo P95:", umbral_rodeo)

Umbral distancia red P95: 2565.3923293443236
Umbral rodeo P95: 14.442797043778945


In [78]:
df_aristas_reales["revisar_distancia_red_alta"] = (
    df_aristas_reales["distancia_red_m"] > umbral_distancia_red
)

df_aristas_reales["revisar_rodeo_alto"] = (
    df_aristas_reales["factor_rodeo"] > umbral_rodeo
)

def clasificar_arista(row):
    if row["revisar_distancia_red_alta"] and row["revisar_rodeo_alto"]:
        return "revisar_distancia_y_rodeo"
    elif row["revisar_rodeo_alto"]:
        return "revisar_rodeo_alto"
    elif row["revisar_distancia_red_alta"]:
        return "revisar_distancia_red_alta"
    else:
        return "ok"

df_aristas_reales["calidad_arista"] = df_aristas_reales.apply(clasificar_arista, axis=1)

df_aristas_reales["calidad_arista"].value_counts()

calidad_arista
ok                            56484
revisar_distancia_y_rodeo      2257
revisar_rodeo_alto              756
revisar_distancia_red_alta      754
Name: count, dtype: int64

In [79]:
# Aristas con mayor factor de rodeo
df_aristas_reales.sort_values("factor_rodeo", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,revisar_distancia_red_alta,revisar_rodeo_alto,calidad_arista
57566,6952,6949,5360984763,307997125,12.250197,9003.287558,camino_osm,734.950412,True,True,revisar_distancia_y_rodeo
46633,1015,1016,338920029,21723233,8.791932,6123.576228,camino_osm,696.499499,True,True,revisar_distancia_y_rodeo
9513,11370,6786,25549913,297767512,13.582513,6412.259469,camino_osm,472.096692,True,True,revisar_distancia_y_rodeo
57567,6952,6950,5360984763,307997125,19.560019,9003.287558,camino_osm,460.290317,True,True,revisar_distancia_y_rodeo
9540,1052,1049,388087148,2493682299,25.653528,11766.287184,camino_osm,458.661561,True,True,revisar_distancia_y_rodeo
57569,6951,6949,5360984763,307997125,19.911916,9003.287558,camino_osm,452.155757,True,True,revisar_distancia_y_rodeo
1162,11373,11374,9823064711,429461526,18.481650,6765.444483,camino_osm,366.062801,True,True,revisar_distancia_y_rodeo
56893,3826,6790,21525883,20953256,6.961135,2341.325133,camino_osm,336.342442,False,True,revisar_rodeo_alto
1163,11373,11375,9823064711,429461526,21.174514,6765.444483,camino_osm,319.508849,True,True,revisar_distancia_y_rodeo
26829,6738,6737,25938853,2537144683,18.511306,5833.573622,camino_osm,315.135710,True,True,revisar_distancia_y_rodeo


In [80]:
# Aristas con mayor distancia real sobre red
df_aristas_reales.sort_values("distancia_red_m", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,revisar_distancia_red_alta,revisar_rodeo_alto,calidad_arista
10268,11496,6856,315244935,2136384930,218.721078,13963.539403,camino_osm,63.841764,True,True,revisar_distancia_y_rodeo
10262,11496,6858,315244935,2136384930,209.732125,13963.539403,camino_osm,66.577971,True,True,revisar_distancia_y_rodeo
10259,11496,6857,315244935,2136384930,208.098671,13963.539403,camino_osm,67.100570,True,True,revisar_distancia_y_rodeo
10272,11496,7118,315244935,2136384930,239.828567,13963.539403,camino_osm,58.223003,True,True,revisar_distancia_y_rodeo
10270,11496,1032,315244935,2136384930,231.140447,13963.539403,camino_osm,60.411493,True,True,revisar_distancia_y_rodeo
10269,11496,7145,315244935,2136384930,219.215723,13963.539403,camino_osm,63.697709,True,True,revisar_distancia_y_rodeo
49876,1035,6857,315244935,2136384930,204.252481,13963.539403,camino_osm,68.364112,True,True,revisar_distancia_y_rodeo
49877,1035,6858,315244935,2136384930,206.291569,13963.539403,camino_osm,67.688367,True,True,revisar_distancia_y_rodeo
49881,1035,6856,315244935,2136384930,214.867578,13963.539403,camino_osm,64.986721,True,True,revisar_distancia_y_rodeo
49888,1035,7118,315244935,2136384930,237.149216,13963.539403,camino_osm,58.880816,True,True,revisar_distancia_y_rodeo


## Construcción grafo

In [81]:
G_medidores = nx.DiGraph()

# Añadir todos los medidores reales como nodos
for _, row in medidores.iterrows():
    G_medidores.add_node(
        int(row["id"]),
        nombre=row["nombre"],
        latitud=float(row["latitud"]),
        longitud=float(row["longitud"]),
        osm_node=int(row["osm_node"]),
        distancia_osm_node_m=float(row["distancia_osm_node_m"])
    )

# Añadir aristas reales calculadas sobre OSM
for _, row in df_aristas_reales.iterrows():
    G_medidores.add_edge(
        int(row["id_origen"]),
        int(row["id_destino"]),
        distancia_directa_m=float(row["distancia_directa_m"]),
        distancia_red_m=float(row["distancia_red_m"]),
        factor_rodeo=float(row["factor_rodeo"]),
        weight=float(row["distancia_red_m"]),
        tipo_conexion=row["tipo_conexion"],
        calidad_arista=row["calidad_arista"]
    )

print("Nodos:", G_medidores.number_of_nodes())
print("Aristas:", G_medidores.number_of_edges())

Nodos: 5072
Aristas: 60251


In [82]:
nodos_con_aristas = set(df_aristas_reales["id_origen"]).union(
    set(df_aristas_reales["id_destino"])
)

nodos_aislados = set(medidores["id"]) - nodos_con_aristas

print("Medidores totales:", medidores["id"].nunique())
print("Medidores con alguna arista:", len(nodos_con_aristas))
print("Medidores aislados:", len(nodos_aislados))

print("Componentes débiles:", nx.number_weakly_connected_components(G_medidores))
print("Componentes fuertes:", nx.number_strongly_connected_components(G_medidores))

Medidores totales: 5072
Medidores con alguna arista: 5060
Medidores aislados: 12
Componentes débiles: 79
Componentes fuertes: 98


In [83]:
medidores_aislados = medidores[
    medidores["id"].isin(nodos_aislados)
].copy()

medidores_aislados[
    ["id", "nombre", "latitud", "longitud", "osm_node", "distancia_osm_node_m"]
].sort_values("distancia_osm_node_m", ascending=False)

,id,nombre,latitud,longitud,osm_node,distancia_osm_node_m
1938,5268,(TACTICO)SALIDA POLIGONO N-S,40.365688,-3.771300,306400716,345.242185
345,10210,PM43041,40.515611,-3.685133,2590772533,101.383315
4388,6584,(TACTICO) SALIDA CUARTEL ARTILLERIA,40.513409,-3.679176,255961297,90.867682
2302,6489,Embajadores - Santa Catalina-Carretera Villave...,40.369021,-3.676004,306101165,90.843345
2761,5160,(TACTICO)JOSE CADALSO S-N(VALLE INCLAN-AV. LAS...,40.382518,-3.771977,26085559,35.193999
872,6923,San Cipriano - Efigencia-Caños San Pedro,40.403851,-3.601293,307534056,20.827393
4743,3528,Tumaco - Tumaco-Tampico,40.445069,-3.635408,114123521,20.281911
675,5298,(TACTICO)ALLARIZ O-E(PROGRESO-AV. CARABANCHEL ...,40.367476,-3.753622,306163498,16.163931
3214,4868,(TACTICO) BATALLA GARELLANO Nº 27 S-N (SIRRACH...,40.455164,-3.793131,292702537,15.418926
3540,10012,(TACTICO) Salida Clinica Lopez Ibor,40.467961,-3.724342,4777894121,13.405010
